In [1]:
import torch
from datasets import load_dataset
import numpy as np
import numba
from tqdm.notebook import tqdm
from collections import Counter
import regex as re
import multiprocessing as mp
from functools import reduce
import operator
from dask import delayed, compute, bag
from functools import partial
import numpy as np

from dask.distributed import Client, get_client

client = Client(n_workers=32, threads_per_worker=1)

In [2]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 32
Total threads: 32,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61885,Workers: 32
Dashboard: http://127.0.0.1:8787/status,Total threads: 32
Started: Just now,Total memory: 16.00 GiB
Comm: tcp://127.0.0.1:61952,Total threads: 1
Dashboard: http://127.0.0.1:61953/status,Memory: 512.00 MiB
Nanny: tcp://127.0.0.1:61888,


In [2]:
tiny_stories = load_dataset("SimpleStories/SimpleStories")

In [3]:
len(tiny_stories['train'])

2115696

In [51]:
n = 10000
small_test_portion = tiny_stories['train'].shuffle(seed=42).select(range(n))

In [52]:
small_test_portion['story']

['Rocket engines roared as they lifted off from Earth. In the cockpit sat a young person, looking out at the blue planet below. The journey was not just through space, but into the heart of who they were. They felt small against the vast sky but also felt a deep sense of wonder. Each star was a new idea, a new piece of them. The ship moved forward, and so did their thoughts.\n\nWhile flying past the moon, they thought about their dreams. Like the moon, dreams can be bright but also far away. They wondered if they could ever reach them. Would they be enough? The ship glided silently, and the stars twinkled like the questions in their mind. "I am a traveler," they thought. "I am made for this." It was a truth they had yet to accept.\n\nThe ship approached a large planet, swirling with colors. It looked like a painting, alive and beautiful. As they got closer, they saw that the planet had storms and calm spots, just like their own life. Each storm was a challenge, and each calm was a mome

In [53]:
tokenizer_string = ' '.join(small_test_portion['story'])
tokenizer_int_list = list(tokenizer_string.encode('utf-8'))
#print(tokenizer_int_list)
len(tokenizer_int_list)

12378796

## BPE

##### Slow python loop version

In [ ]:
class PythonBPE():
    def __init__(self):
        self.BPE_dict = {}
        self.inverse_BPE_dict = {}

    def train(self, train_string, target_dict_size=1000, new_token=None, pbar=True):
        int_list = list(train_string.encode('utf-8'))
        initial_dictionary_size = len(set(int_list))
        initial_tokens = len(int_list)
        if new_token is None:
            new_token = max(int_list) + 1
        replacement_dict = {}
        if pbar:
            iter = tqdm(range(target_dict_size - initial_dictionary_size))
        else:
            iter = range(target_dict_size - initial_dictionary_size)
        for i in iter:
            bigram = self.most_common_bigram(int_list)
            int_list = self.replace_bigrams(int_list, bigram, new_token)
            replacement_dict[bigram] = new_token
            new_token += 1

        self.BPE_dict = replacement_dict
        self.inverse_BPE_dict = {value: key for key, value in reversed(replacement_dict.items())}

        print(f'Dictionary size:\nInitial: {initial_dictionary_size}\nNew: {target_dict_size}')
        print(f'\nInput tokens:\nInitial: {initial_tokens}\nNew: {len(int_list)}')

    def encode(self, string, pbar=True):
        int_list = list(string.encode('utf-8'))
        if pbar:
            iter = tqdm(self.BPE_dict.items())
        else:
            iter = self.BPE_dict.items()
        for bigram, token in iter:
            int_list = self.replace_bigrams(int_list, bigram, token)
        return int_list

    def decode(self, tokens, pbar=True):
        new_tokens = tokens.copy()
        if pbar:
            iter = tqdm(self.inverse_BPE_dict.items())
        else:
            iter = self.inverse_BPE_dict.items()
        for token, replacement in iter:
            new_tokens = self.insert_bigrams(new_tokens, token, replacement)
        return bytes(new_tokens).decode('utf-8', errors='replace')

    @staticmethod
    def most_common_bigram(int_list):
        counter = {}
        for i in range(len(int_list) - 1):
            bigram = tuple(int_list[i:i+2])
            counter_val = counter.get(bigram, 0)
            counter_val += 1
            counter[bigram] = counter_val
        most_common = sorted(counter, key=counter.get, reverse=True)[0]
        return most_common

    @staticmethod
    def replace_bigrams(int_list, bigram, replace):
        new_list =  []
        i = 0
        while True:
            if tuple(int_list[i:i+2]) == bigram:
                new_list.append(replace)
                i += 2
            else:
                new_list.append(int_list[i])
                i += 1
            if i == len(int_list):
                return new_list

    @staticmethod
    def insert_bigrams(int_list, token, bigram):
        new_list = []
        i = 0
        while True:
            if int_list[i] == token:
                new_list += list(bigram)
            else:
                new_list.append(int_list[i])
            i += 1
            if i == len(int_list):
                return new_list


In [ ]:
bpe = PythonBPE()
bpe.train(tokenizer_string, target_dict_size=2000)

In [ ]:
bpe.decode(bpe.encode(tokenizer_string)) == tokenizer_string

In [ ]:
tokens = [bpe.decode([token], pbar=False) for token in set(bpe.encode(tokenizer_string))]

In [ ]:
sorted(tokens, key=len, reverse=True)

In [ ]:
test_string = tiny_stories['train'][0]['story']
test_int_list = list(test_string.encode('utf-8'))
char_length = len(test_int_list)
char_length

In [ ]:
len(bpe.encode(test_string))

In [ ]:
tokens

## Improve the efficiency

In [ ]:
import numpy as np

In [ ]:
class BPE():
    def __init__(self):
        self.BPE_dict = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def train(self, train_string, target_dict_size=1000, new_token=None, pbar=True):
        int_list = list(train_string.encode('utf-8'))
        initial_dictionary_size = len(set(int_list))
        initial_tokens = len(int_list)
        if new_token is None:
            new_token = 256
        replacement_dict = {}
        if pbar:
            iter = tqdm(range(target_dict_size - initial_dictionary_size))
        else:
            iter = range(target_dict_size - initial_dictionary_size)
        for i in iter:
            bigram = self.most_common_bigram(int_list)
            int_list = self.replace_bigrams(int_list, bigram, new_token)
            replacement_dict[bigram] = new_token
            new_token += 1

        self.BPE_dict = replacement_dict
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        for (token_1, token_2), new_token in replacement_dict.items():
            self.vocab[new_token] = self.vocab[token_1] + self.vocab[token_2]

        print(f'Dictionary size:\nInitial: {initial_dictionary_size}\nNew: {target_dict_size}')
        print(f'\nInput tokens:\nInitial: {initial_tokens}\nNew: {len(int_list)}')

    def encode(self, string, pbar=True):
        int_list = list(string.encode('utf-8'))
        if pbar:
            iter = tqdm(self.BPE_dict.items())
        else:
            iter = self.BPE_dict.items()
        for bigram, token in iter:
            int_list = self.replace_bigrams(int_list, bigram, token)
        return int_list

    def decode(self, tokens, pbar=True):
        if pbar:
            iter = tqdm(tokens)
        else:
            iter = tokens
        return (b"".join(self.vocab[token] for token in iter)).decode('utf-8', errors='replace')

    @staticmethod
    def most_common_bigram(int_list):
        bigrams = Counter(zip(int_list, int_list[1:]))
        return bigrams.most_common(1)[0][0]

    @staticmethod
    def replace_bigrams(int_list, bigram, replace):
        new_list =  []
        i = 0
        while True:
            if tuple(int_list[i:i+2]) == bigram:
                new_list.append(replace)
                i += 2
            else:
                new_list.append(int_list[i])
                i += 1
            if i == len(int_list):
                return new_list

    @staticmethod
    def insert_bigrams(int_list, token, bigram):
        new_list = []
        i = 0
        while True:
            if int_list[i] == token:
                new_list += list(bigram)
            else:
                new_list.append(int_list[i])
            i += 1
            if i == len(int_list):
                return new_list


### Add Regex Splitting

In [10]:

# Pulled from minbpe karpathy
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexBPE():
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        self.pattern = pattern
        self.compiled_pattern = re.compile(pattern)
        self.merge_dict = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def chunk_input(self, input):
        return re.findall(self.compiled_pattern, input)

    def train(self, train_string, num_merges=1000, new_token=None, pbar=True, npartitions=32):

        chunks = self.chunk_input(train_string)

        chunk_bag = bag.from_sequence(chunks, npartitions=npartitions)

        encoded_chunks = chunk_bag.map(self.encode_chunk).persist()


        counters = encoded_chunks.map(lambda x: Counter(zip(x, x[1:])))
        total_counter = counters.fold(binop=operator.add).compute()

        if new_token is None:
            new_token = 257

        for i in tqdm(range(num_merges)):
            bigram_to_replace = total_counter.most_common(1)[0][0]

            replacement_func = partial(self.replace_bigram_with_counter, bigram=bigram_to_replace, new_token=new_token)
            new_chunks_and_counters = encoded_chunks.map(replacement_func).persist()

            old_encoded_chunks = encoded_chunks
            encoded_chunks = new_chunks_and_counters.map(lambda x: x[0])#.filter(lambda x: len(x)>1)
            counter_update = new_chunks_and_counters.map(lambda x: x[1]).fold(binop=self.counter_sum_keep_negative).compute()
            del old_encoded_chunks

            self.merge_dict[bigram_to_replace] = new_token

            total_counter += counter_update
            self.vocab[new_token] = self.vocab[bigram_to_replace[0]] + self.vocab[bigram_to_replace[1]]
            new_token += 1

    def encode(self, string, pbar=True):
        int_list = list(string.encode('utf-8'))
        if pbar:
            iter = tqdm(self.merge_dict.items())
        else:
            iter = self.merge_dict.items()
        for bigram, token in iter:
            int_list = self.replace_bigrams(int_list, bigram, token)
        return int_list

    def encode(self, string, pbar=True):
        chunks = self.chunk_input(string)
        chunk_bag = bag.from_sequence(chunks, npartitions=32)
        encoded_chunks = chunk_bag.map_partitions(self.encode_chunk).persist()

        for bigram, token in self.merge_dict.items():
            replacement_func = partial(self.replace_bigram, bigram=bigram, new_token=token)
            encoded_chunks = encoded_chunks.map(replacement_func).persist()

        return encoded_chunks.fold(binop=operator.add).compute()

    def decode(self, tokens, pbar=True):
        if pbar:
            iter = tqdm(tokens)
        else:
            iter = tokens
        return (b"".join(self.vocab[token] for token in iter)).decode('utf-8', errors='replace')

    # Just named functions so can see what's happening in dask.
    @staticmethod
    def extract_chunks(output):
        return output[0]

    @staticmethod
    def extract_counters(output):
        return output[1]

    @staticmethod
    def filter_chunks(output):
        return len(output) > 1

    @staticmethod
    def encode_chunk(chunk):
        return list(chunk.encode('utf-8'))

    @staticmethod
    def replace_bigram_with_counter(chunk, bigram, new_token):
        new_chunk = []
        i = 0
        counter_update = Counter()

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                if i > 0:
                    counter_update[(chunk[i-1], chunk[i])] -= 1
                    counter_update[(chunk[i-1], new_token)] += 1
                if i + 2 < len(chunk):
                    counter_update[(chunk[i+1], chunk[i+2])] -= 1
                    counter_update[(new_token, chunk[i+2])] += 1

                counter_update[bigram] -= 1
                new_chunk.append(new_token)
                i+=2
            else:
                new_chunk.append(chunk[i])
                i+=1
        return new_chunk, counter_update

    @staticmethod
    def replace_bigram(chunk, bigram, new_token):
        new_chunk = []
        i = 0

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                new_chunk.append(new_token)
                i+=2
            else:
                new_chunk.append(chunk[i])
                i+=1
        return new_chunk

    @staticmethod
    def counter_sum_keep_negative(a, b):
        a = a.copy()
        for k, v in b.items():
            a[k] += v
        return a




In [11]:
rbpe = RegexBPE()
rbpe.train(tokenizer_string, npartitions=64, num_merges=1000)

/home/z5345028/miniconda3/envs/hamiltonian-learning/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 18.98 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

rbpe.merge_dict

In [ ]:
rbpe.vocab

In [ ]:
import pickle
with open('bpe_dict.pickle', 'wb') as file:
    pickle.dump(rbpe.merge_dict, file)

In [14]:
# Pulled from minbpe karpathy
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

def count_bigrams(input):
    return Counter(zip(input, input[1:]))

class RegexBPE():
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        self.pattern = pattern
        self.compiled_pattern = re.compile(pattern)
        self.merge_dict = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def chunk_input(self, input):
        return re.findall(self.compiled_pattern, input)

    def train(self, train_string, num_merges=1000, new_token=None, pbar=True, npartitions=32):

        chunks = self.chunk_input(train_string)

        chunk_bag = bag.from_sequence(chunks, npartitions=npartitions)

        encoded_chunks = chunk_bag.map(self.encode_chunk).persist()


        counters = encoded_chunks.map(lambda x: Counter(zip(x, x[1:])))
        total_counter = counters.fold(binop=operator.add).compute()

        if new_token is None:
            new_token = 257

        for i in tqdm(range(num_merges)):
            if not total_counter:
                print("Warning: No more bigrams to merge")
                break

            bigram_to_replace = total_counter.most_common(1)[0][0]

            replacement_func = partial(self.replace_bigram_with_counter, bigram=bigram_to_replace, new_token=new_token)
            new_chunks_and_counters = encoded_chunks.map(replacement_func)


            new_encoded_chunks = new_chunks_and_counters.map(lambda x: x[0])#.filter(lambda x: len(x)>1)
            counter_updates = new_chunks_and_counters.map(lambda x: x[1])

            counter_update = counter_updates.reduction(perpartition=lambda seq: reduce(RegexBPE.counter_sum_keep_negative, seq),
                                                        aggregate=lambda seq: reduce(RegexBPE.counter_sum_keep_negative, seq), out_type=Counter).compute()

            print(counter_update)


            self.merge_dict[bigram_to_replace] = new_token
            self.vocab[new_token] = self.vocab[bigram_to_replace[0]] + self.vocab[bigram_to_replace[1]]

            total_counter += counter_update
            total_counter = Counter({k: v for k, v in total_counter.items() if v > 0})

            encoded_chunks = new_encoded_chunks.persist()
            new_token += 1

    def encode(self, string, pbar=True):
        int_list = list(string.encode('utf-8'))
        if pbar:
            iter = tqdm(self.merge_dict.items())
        else:
            iter = self.merge_dict.items()
        for bigram, token in iter:
            int_list = self.replace_bigrams(int_list, bigram, token)
        return int_list

    def encode(self, string, pbar=True):
        chunks = self.chunk_input(string)
        chunk_bag = bag.from_sequence(chunks, npartitions=32)
        encoded_chunks = chunk_bag.map_partitions(self.encode_chunk).persist()

        for bigram, token in self.merge_dict.items():
            replacement_func = partial(self.replace_bigram, bigram=bigram, new_token=token)
            encoded_chunks = encoded_chunks.map(replacement_func).persist()

        return encoded_chunks.fold(binop=operator.add).compute()

    def decode(self, tokens, pbar=True):
        if pbar:
            iter = tqdm(tokens)
        else:
            iter = tokens
        return (b"".join(self.vocab[token] for token in iter)).decode('utf-8', errors='replace')

    # Just named functions so can see what's happening in dask.
    @staticmethod
    def extract_chunks(output):
        return output[0]

    @staticmethod
    def extract_counters(output):
        return output[1]

    @staticmethod
    def filter_chunks(output):
        return len(output) > 1

    @staticmethod
    def encode_chunk(chunk):
        return list(chunk.encode('utf-8'))

    @staticmethod
    def replace_bigram_with_counter(chunk, bigram, new_token):
        new_chunk = []
        i = 0
        counter_update = Counter()

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                if i > 0:
                    counter_update[(chunk[i-1], chunk[i])] -= 1
                    counter_update[(chunk[i-1], new_token)] += 1
                if i + 2 < len(chunk):
                    counter_update[(chunk[i+1], chunk[i+2])] -= 1
                    counter_update[(new_token, chunk[i+2])] += 1

                counter_update[bigram] -= 1
                new_chunk.append(new_token)
                i+=2
            else:
                new_chunk.append(chunk[i])
                i+=1
        return new_chunk, counter_update

    @staticmethod
    def replace_bigram(chunk, bigram, new_token):
        new_chunk = []
        i = 0

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                new_chunk.append(new_token)
                i+=2
            else:
                new_chunk.append(chunk[i])
                i+=1
        return new_chunk

    @staticmethod
    def counter_sum_keep_negative(a, b):
        a = a.copy()
        for k, v in b.items():
            a[k] += v
        return a





In [15]:
rbpe = RegexBPE()
rbpe.train(tokenizer_string, npartitions=64, num_merges=1000)

/home/z5345028/miniconda3/envs/hamiltonian-learning/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 19.00 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


  0%|          | 0/1000 [00:00<?, ?it/s]

[(116, 104), (116, 257), (101, 121), (257, 121), (104, 101), (84, 104), (84, 257), (32, 104), (32, 257), (101, 97), (257, 97), (101, 109), (257, 109), (101, 105), (257, 105), (99, 104), (99, 257), (101, 100), (257, 100), (101, 101), (257, 101), (119, 104), (119, 257), (101, 114), (257, 114), (103, 104), (103, 257), (101, 110), (257, 110), (83, 104), (83, 257), (115, 104), (115, 257), (101, 108), (257, 108), (87, 104), (87, 257), (101, 115), (257, 115), (112, 104), (112, 257), (97, 104), (97, 257), (67, 104), (67, 257), (101, 99), (257, 99), (114, 104), (114, 257), (101, 102), (257, 102), (101, 104), (101, 257), (101, 119), (257, 119), (45, 104), (45, 257), (111, 104), (111, 257), (101, 116), (257, 116), (80, 104), (80, 257), (110, 104), (110, 257), (65, 104), (65, 257), (108, 104), (108, 257), (101, 111), (257, 111)]


AttributeError: 'list' object has no attribute 'items'

In [ ]:
print(rbpe.decode(rbpe.encode(tokenizer_string)) == tokenizer_string)
print(len(list(tokenizer_string.encode('utf-8'))))
print(len(rbpe.encode(tokenizer_string)))

In [ ]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexBPE:
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        self.pattern = pattern
        self.compiled_pattern = re.compile(pattern)
        self.merge_dict = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def chunk_input(self, input_str):
        return re.findall(self.compiled_pattern, input_str)

    def train(self, train_string, num_merges=1000, new_token=None, pbar=True):
        chunks = self.chunk_input(train_string)
        encoded_chunks = [self.encode_chunk(chunk) for chunk in chunks]

        # Count all bigrams
        counters = [Counter(zip(chunk, chunk[1:])) for chunk in encoded_chunks]
        total_counter = self.counter_sum_keep_negative(*counters)

        if new_token is None:
            new_token = 257

        merge_iter = tqdm(range(num_merges)) if pbar else range(num_merges)
        for _ in merge_iter:
            if not total_counter:
                break  # No more bigrams to merge
            bigram_to_replace = total_counter.most_common(1)[0][0]

            new_encoded_chunks = []
            counter_deltas = []

            for chunk in encoded_chunks:
                new_chunk, delta = self.replace_bigram(chunk, bigram_to_replace, new_token)
                new_encoded_chunks.append(new_chunk)
                counter_deltas.append(delta)

            encoded_chunks = new_encoded_chunks
            delta_counter = self.counter_sum_keep_negative(*counter_deltas)
            total_counter = self.counter_sum_keep_negative(total_counter, delta_counter)

            self.merge_dict[bigram_to_replace] = new_token
            new_token += 1

        # Update vocab
        for (token1, token2), tok in self.merge_dict.items():
            self.vocab[tok] = self.vocab[token1] + self.vocab[token2]

    def encode(self, string, pbar=True):
        int_list = list(string.encode('utf-8'))
        merge_iter = tqdm(self.merge_dict.items()) if pbar else self.merge_dict.items()
        for bigram, token in merge_iter:
            int_list = self.replace_bigrams(int_list, bigram, token)
        return int_list

    def decode(self, tokens, pbar=True):
        iter_tokens = tqdm(tokens) if pbar else tokens
        return b"".join(self.vocab.get(tok, b"") for tok in iter_tokens).decode('utf-8', errors='replace')

    @staticmethod
    def encode_chunk(chunk):
        return list(chunk.encode('utf-8'))

    @staticmethod
    def replace_bigram(chunk, bigram, new_token):
        new_chunk = []
        i = 0
        delta = Counter()

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                if i > 0:
                    delta[(chunk[i-1], chunk[i])] -= 1
                    delta[(chunk[i-1], new_token)] += 1
                if i + 2 < len(chunk):
                    delta[(chunk[i+1], chunk[i+2])] -= 1
                    delta[(new_token, chunk[i+2])] += 1
                delta[bigram] -= 1
                new_chunk.append(new_token)
                i += 2
            else:
                new_chunk.append(chunk[i])
                i += 1
        return new_chunk, delta

    @staticmethod
    def replace_bigrams(seq, bigram, replacement):
        result = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == bigram:
                result.append(replacement)
                i += 2
            else:
                result.append(seq[i])
                i += 1
        return result

    @staticmethod
    def counter_sum_keep_negative(*counters):
        result = Counter()
        for c in counters:
            for k, v in c.items():
                result[k] += v
        return result

In [ ]:
rbpe = RegexBPE()
%time rbpe.train(tokenizer_string)

In [ ]:
rbpe.vocab

### Use Numba

In [9]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class NumbaRegexBPE():
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        self.pattern = pattern
        self.compiled_pattern = re.compile(pattern)
        self.merge_dict = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def chunk_input(self, input):
        return re.findall(self.compiled_pattern, input)


    def train(self, train_string, num_merges=1000, new_token=None, pbar=True, npartitions=32):

        chunks = self.chunk_input(train_string)

        encoded_chunks = map(self.encode_chunk, chunks)


        counters = encoded_chunks.map(lambda x: Counter(zip(x, x[1:])))
        total_counter = counters.fold(binop=operator.add).compute()

        if new_token is None:
            new_token = 257

        for i in tqdm(range(num_merges)):
            bigram_to_replace = total_counter.most_common(1)[0][0]

            replacement_func = partial(self.replace_bigram_with_counter, bigram=bigram_to_replace, new_token=new_token)
            new_chunks_and_counters = encoded_chunks.map(replacement_func).persist()

            old_encoded_chunks = encoded_chunks
            encoded_chunks = new_chunks_and_counters.map(lambda x: x[0])#.filter(lambda x: len(x)>1)
            counter_update = new_chunks_and_counters.map(lambda x: x[1]).fold(binop=self.counter_sum_keep_negative).compute()
            del old_encoded_chunks

            self.merge_dict[bigram_to_replace] = new_token

            total_counter += counter_update
            self.vocab[new_token] = self.vocab[bigram_to_replace[0]] + self.vocab[bigram_to_replace[1]]
            new_token += 1

    def encode(self, string, pbar=True):
        int_list = list(string.encode('utf-8'))
        if pbar:
            iter = tqdm(self.merge_dict.items())
        else:
            iter = self.merge_dict.items()
        for bigram, token in iter:
            int_list = self.replace_bigrams(int_list, bigram, token)
        return int_list

    def encode(self, string, pbar=True):
        chunks = self.chunk_input(string)
        chunk_bag = bag.from_sequence(chunks, npartitions=32)
        encoded_chunks = chunk_bag.map_partitions(self.encode_chunk).persist()

        for bigram, token in self.merge_dict.items():
            replacement_func = partial(self.replace_bigram, bigram=bigram, new_token=token)
            encoded_chunks = encoded_chunks.map(replacement_func).persist()

        return encoded_chunks.fold(binop=operator.add).compute()

    def decode(self, tokens, pbar=True):
        if pbar:
            iter = tqdm(tokens)
        else:
            iter = tokens
        return (b"".join(self.vocab[token] for token in iter)).decode('utf-8', errors='replace')

    # Just named functions so can see what's happening in dask.
    @staticmethod
    def extract_chunks(output):
        return output[0]

    @staticmethod
    def extract_counters(output):
        return output[1]

    @staticmethod
    def filter_chunks(output):
        return len(output) > 1

    @staticmethod
    def encode_chunk(chunk):
        return np.frombuffer(chunk.encode('utf-8'), dtype=np.uint8)

    @staticmethod
    def replace_bigram_with_counter(chunk, bigram, new_token):
        new_chunk = []
        i = 0
        counter_update = Counter()

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                if i > 0:
                    counter_update[(chunk[i-1], chunk[i])] -= 1
                    counter_update[(chunk[i-1], new_token)] += 1
                if i + 2 < len(chunk):
                    counter_update[(chunk[i+1], chunk[i+2])] -= 1
                    counter_update[(new_token, chunk[i+2])] += 1

                counter_update[bigram] -= 1
                new_chunk.append(new_token)
                i+=2
            else:
                new_chunk.append(chunk[i])
                i+=1
        return new_chunk, counter_update

    @staticmethod
    def replace_bigram(chunk, bigram, new_token):
        new_chunk = []
        i = 0

        while i < len(chunk):
            if i < len(chunk) - 1 and (chunk[i], chunk[i+1]) == bigram:
                new_chunk.append(new_token)
                i+=2
            else:
                new_chunk.append(chunk[i])
                i+=1
        return new_chunk

    @staticmethod
    def counter_sum_keep_negative(a, b):
        a = a.copy()
        for k, v in b.items():
            a[k] += v
        return a


In [54]:
nrbpe = NumbaRegexBPE()
chunked_input = nrbpe.chunk_input(tokenizer_string)

In [62]:
from joblib import Parallel, delayed

def encode_chunk_numpy(chunk):
    return np.frombuffer(chunk.encode('utf-8'), dtype=np.uint8)

def encode_chunk_list(chunk):
    return list(chunk.encode('utf-8'))

def encode_chunks_numpy(chunks):
    return [encode_chunk_numpy(chunk) for chunk in chunks]

def encode_chunks_list(chunks):
    return [encode_chunk_list(chunk) for chunk in chunks]

def encode_chunks_map(chunks):
    return list(map(encode_chunk_numpy, chunks))

def encode_chunks_parallel(chunks):
    return Parallel(n_jobs=-1)(delayed(encode_chunk_numpy)(chunk) for chunk in chunks)


In [63]:
%timeit encode_chunks_numpy(chunked_input)
%timeit encode_chunks_list(chunked_input)
%timeit encode_chunks_map(chunked_input)
%timeit encode_chunks_parallel(chunked_input)

1.03 s ± 12.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
401 ms ± 9.08 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
1.03 s ± 12.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


KeyboardInterrupt: 

In [58]:
encoded_chunks = encode_chunks_numpy(chunked_input)


In [59]:
encoded_chunks

[array([ 82, 111,  99, 107, 101, 116], dtype=uint8),
 array([ 32, 101, 110, 103, 105, 110, 101, 115], dtype=uint8),
 array([ 32, 114, 111,  97, 114, 101, 100], dtype=uint8),
 array([ 32,  97, 115], dtype=uint8),
 array([ 32, 116, 104, 101, 121], dtype=uint8),
 array([ 32, 108, 105, 102, 116, 101, 100], dtype=uint8),
 array([ 32, 111, 102, 102], dtype=uint8),
 array([ 32, 102, 114, 111, 109], dtype=uint8),
 array([ 32,  69,  97, 114, 116, 104], dtype=uint8),
 array([46], dtype=uint8),
 array([ 32,  73, 110], dtype=uint8),
 array([ 32, 116, 104, 101], dtype=uint8),
 array([ 32,  99, 111,  99, 107, 112, 105, 116], dtype=uint8),
 array([ 32, 115,  97, 116], dtype=uint8),
 array([32, 97], dtype=uint8),
 array([ 32, 121, 111, 117, 110, 103], dtype=uint8),
 array([ 32, 112, 101, 114, 115, 111, 110], dtype=uint8),
 array([44], dtype=uint8),
 array([ 32, 108, 111, 111, 107, 105, 110, 103], dtype=uint8),
 array([ 32, 111, 117, 116], dtype=uint8),
 array([ 32,  97, 116], dtype=uint8),
 array([ 32

In [64]:
chunked_input

['Rocket',
 ' engines',
 ' roared',
 ' as',
 ' they',
 ' lifted',
 ' off',
 ' from',
 ' Earth',
 '.',
 ' In',
 ' the',
 ' cockpit',
 ' sat',
 ' a',
 ' young',
 ' person',
 ',',
 ' looking',
 ' out',
 ' at',
 ' the',
 ' blue',
 ' planet',
 ' below',
 '.',
 ' The',
 ' journey',
 ' was',
 ' not',
 ' just',
 ' through',
 ' space',
 ',',
 ' but',
 ' into',
 ' the',
 ' heart',
 ' of',
 ' who',
 ' they',
 ' were',
 '.',
 ' They',
 ' felt',
 ' small',
 ' against',
 ' the',
 ' vast',
 ' sky',
 ' but',
 ' also',
 ' felt',
 ' a',
 ' deep',
 ' sense',
 ' of',
 ' wonder',
 '.',
 ' Each',
 ' star',
 ' was',
 ' a',
 ' new',
 ' idea',
 ',',
 ' a',
 ' new',
 ' piece',
 ' of',
 ' them',
 '.',
 ' The',
 ' ship',
 ' moved',
 ' forward',
 ',',
 ' and',
 ' so',
 ' did',
 ' their',
 ' thoughts',
 '.\n\n',
 'While',
 ' flying',
 ' past',
 ' the',
 ' moon',
 ',',
 ' they',
 ' thought',
 ' about',
 ' their',
 ' dreams',
 '.',
 ' Like',
 ' the',
 ' moon',
 ',',
 ' dreams',
 ' can',
 ' be',
 ' bright',
 ' but',
 